# Czech UGS Storage Obligations — Monthly Regulatory Requirements

**Question this notebook answers**
What volume of working gas must be held in Czech UGS at the **start of each
heating-season month** (Oct–Mar) so that a 30-day peak-demand stress event
(as defined in EU Regulation 2017/1938 Art. 6) can be survived, at each
import-risk scenario (S1–S5)?

**Regulatory framing**
Article 6 of the Gas Security of Supply Regulation requires that protected
customers be supplied during:
- A **7-day extreme cold spell** at the 1-in-20 daily peak demand (`R.max.den`).
- A broader **30-day period** whose total consumption is given by `r_30dnu`.

The Czech implementation pins this 30-day stress period to calendar months.
The two-tier demand profile: 7 days at `R.max.den` followed by 23 days at
the residual average implied by `r_30dnu`.

**Key design choices**
- Monthly obligations are **independent stress tests** — no carry-forward between months.
- Compliance checked at **month-start checkpoints** only (Oct 1, Nov 1, …, Mar 1).
- Withdrawal-rate constraint applied at each simulation step as fill depletes.
- **End-of-season reserve:** ~0.5 TWh at Mar 31 recommended as a separate instrument.

*All analytical logic lives in `bsd`; this notebook calls the library and renders results.*

---
## 0. Setup

In [ ]:
%cd ..
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import bsd
import bsd.jvs as jvs

jvs.apply_style(theme="light", context="notebook")
%matplotlib inline

SCENARIOS = bsd.DEFAULT_SCENARIOS_DICT

# ── Withdrawal curve (Branch 1: 50/50 blend, 10% haircut) ──
WC_ASSUMPTION       = "blend"
WC_BLEND_WEIGHT_ENG = 0.50   # Branch 1 default — see docs/wc_blend_decision.md
IMPORT_ASSUMPTION   = "p99_cold"   # controls which ceiling is shown in supplementary table

wc_curves = bsd.fit_withdrawal_curves()
wc_func = wc_curves.blend(eng_weight=WC_BLEND_WEIGHT_ENG)

print(f"WC assumption: {WC_ASSUMPTION}  (eng_weight={WC_BLEND_WEIGHT_ENG:.0%})")
print(f"WC at 20%: {wc_func(20):.1f}  40%: {wc_func(40):.1f}  60%: {wc_func(60):.1f}  80%: {wc_func(80):.1f}  GWh/d")

---
## 1. Two-tier demand profile

In [ ]:
prof = bsd.load_demand_profile()
peak_demand     = prof.peak
residual_demand = prof.residual

bsd.demand_profile_table(prof)

---
## 2. Monthly import percentiles per scenario

In [ ]:
monthly_imports = bsd.compute_monthly_imports(SCENARIOS)
p99_cold, p99_uncond = bsd.compute_p99_imports()

IMPORT_CEILING = {"p99_cold": p99_cold,
                  "p99_unconditional": p99_uncond}[IMPORT_ASSUMPTION]

imp_df = bsd.import_table(monthly_imports, p99_cold, p99_uncond)
print(f"Active ceiling assumption: {IMPORT_ASSUMPTION}")
imp_df

---
## 3. Withdrawal-curve reference

Curves fitted by `bsd.fit_withdrawal_curves()` — see `bsd/withdrawal.py` for the
isotonic regression and engineering-curve logic.

In [ ]:
print(f"WC_CURRENT: {wc_curves.wc_current:.1f} GWh/d")
print(f"Blend {WC_BLEND_WEIGHT_ENG:.0%}/{1-WC_BLEND_WEIGHT_ENG:.0%} eng/emp at key fill levels:")
for fp in [20, 40, 60, 80]:
    print(f"  {fp}%: {wc_func(fp):.1f} GWh/d  "
          f"(empirical: {wc_curves.empirical(fp):.1f},  "
          f"engineering: {wc_curves.engineering(fp):.1f})")

---
## 4. Main results — minimum starting fill per month per scenario

`bsd.run_all_months` loops over all scenarios and months, calling
`bsd.min_start_fill_month` (bisection) and `bsd.simulate_month` for each.

In [ ]:
results = bsd.run_all_months(
    monthly_imports, wc_func, peak_demand, residual_demand, SCENARIOS,
)

oblig_twh = bsd.obligations_table(results, SCENARIOS)
print("Minimum required starting fill (TWh) per month per scenario")
oblig_twh

In [ ]:
# Fill-% version
oblig_pct = pd.DataFrame(
    {key: {bsd.MONTH_NAMES[m]: (
        None if results[key][m]["start_fill_pct"] is None
        else round(results[key][m]["start_fill_pct"], 1)
    ) for m in bsd.MONTH_ORDER}
     for key in SCENARIOS}
)
oblig_pct.columns = [SCENARIOS[k].label for k in SCENARIOS]
print("Minimum required starting fill (% of capacity) per month per scenario")
oblig_pct

In [ ]:
# Binding constraint table
binding_table = pd.DataFrame(
    {key: {bsd.MONTH_NAMES[m]: results[key][m]["binding"] for m in bsd.MONTH_ORDER}
     for key in SCENARIOS}
)
binding_table.columns = [SCENARIOS[k].label for k in SCENARIOS]
print("Binding constraint per month per scenario")
binding_table

---
## 5. Charts

In [ ]:
# Heatmap: obligation (TWh) by month × scenario
fig, ax = plt.subplots(figsize=(9, 3.5))
data_arr = oblig_twh.values.astype(float)
im = ax.imshow(data_arr.T, aspect="auto", cmap="YlOrRd", vmin=0, vmax=bsd.CAPACITY_TWH)
ax.set_xticks(range(len(bsd.MONTH_ORDER)))
ax.set_xticklabels([bsd.MONTH_NAMES[m] for m in bsd.MONTH_ORDER])
ax.set_yticks(range(len(SCENARIOS)))
ax.set_yticklabels([SCENARIOS[k].label for k in SCENARIOS])
for i, m in enumerate(bsd.MONTH_ORDER):
    for j, key in enumerate(SCENARIOS):
        v = results[key][m]["start_fill_TWh"]
        txt = f"{v:.1f}" if v is not None else "\u2014"
        ax.text(i, j, txt, ha="center", va="center", fontsize=8.5,
                color="white" if (v or 0) > bsd.CAPACITY_TWH * 0.55 else "#333333")
plt.colorbar(im, ax=ax, label="TWh", fraction=0.03)
ax.set_title("Monthly storage obligation (TWh) — minimum fill at month start")
fig.tight_layout()
fig.savefig("figs/storage_obligations_heatmap.png", bbox_inches="tight")
fig;

In [ ]:
# Grouped bar: all months × scenarios
fig, ax = plt.subplots(figsize=(11, 4.5))
n_months, n_scen, width = len(bsd.MONTH_ORDER), len(SCENARIOS), 0.14
xs = np.arange(n_months)

for j, (key, sc) in enumerate(SCENARIOS.items()):
    vals = [results[key][m]["start_fill_TWh"] or 0.0 for m in bsd.MONTH_ORDER]
    offset = (j - n_scen / 2 + 0.5) * width
    ax.bar(xs + offset, vals, width=width, color=sc.color, alpha=0.85, label=sc.label)

ax.axhline(bsd.CAPACITY_TWH, color="black", lw=1.0, ls="--",
           label=f"Total capacity ({bsd.CAPACITY_TWH:.1f} TWh)")
ax.set_xticks(xs)
ax.set_xticklabels([bsd.MONTH_NAMES[m] for m in bsd.MONTH_ORDER])
ax.set_ylabel("Minimum required fill (TWh)")
ax.set_title("Monthly storage obligations by scenario")
ax.legend(fontsize=8, ncol=3)
ax.set_ylim(0, bsd.CAPACITY_TWH * 1.08)
jvs.grid(ax=ax)
fig.tight_layout()
fig.savefig("figs/storage_obligations_bar.png", bbox_inches="tight")
fig;

In [ ]:
# Fill trajectories for January under each scenario
fig, ax = plt.subplots(figsize=(8, 4))

for key, sc in SCENARIOS.items():
    info = results[key][1]  # January
    traj = info["sim"]["fill_trajectory"]
    f    = info["start_fill_pct"]
    ax.plot(range(len(traj)), traj, color=sc.color, lw=2,
            label=f"{sc.label}  start={f:.1f}%" if f is not None else sc.label)

ax.axvline(bsd.STRESS_PEAK_DAYS, color="grey", lw=0.8, ls=":", alpha=0.7)
ax.text(bsd.STRESS_PEAK_DAYS + 0.3, ax.get_ylim()[1] * 0.97, "peak \u2192 residual",
        fontsize=7.5, color="grey", va="top")
ax.axhline(20, color="grey", lw=0.6, ls="--", alpha=0.6,
           label="Low-confidence WC zone (<20%)")
ax.set_xlabel("Day of stress period")
ax.set_ylabel("Fill level (%)")
ax.set_title("January: fill trajectory over 30-day stress period (from minimum required start)")
ax.legend(fontsize=8)
jvs.grid(ax=ax)
fig.tight_layout()
fig.savefig("figs/storage_obligations_jan_trajectories.png", bbox_inches="tight")
fig;

---
## 6. Regulatory summary table

Primary regulatory output: minimum TWh at month start per scenario.
S2 (P20 imports) is the **recommended planning anchor**.

In [ ]:
rows = []
for m in bsd.MONTH_ORDER:
    row = {"Month": bsd.MONTH_NAMES[m]}
    for key, sc in SCENARIOS.items():
        info = results[key][m]
        v, b = info["start_fill_TWh"], info["binding"]
        if v is None:
            row[sc.label] = "infeasible"
        elif v == 0.0:
            row[sc.label] = "0.00"
        else:
            marker = " \u2605" if b == "withdrawal rate" else ""
            row[sc.label] = f"{v:.2f}{marker}"
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("Month")
print("\u2605 withdrawal-rate binding  (blank = volume binding)")
print(f"Czech UGS total capacity: {bsd.CAPACITY_TWH} TWh")
print("Recommended end-of-season operational floor (separate instrument): ~0.5 TWh at Mar 31\n")
summary_df

---
## 7. Ceiling-import benchmark (P99, cold days)

In [ ]:
# Obligations under ceiling imports (all scenarios)
ceiling_rows = []
for key, sc in SCENARIOS.items():
    imp_ceiling_d = {m: float(IMPORT_CEILING[m]) for m in bsd.MONTH_ORDER}
    ceiling_imp_series = {k: pd.Series(imp_ceiling_d) for k in SCENARIOS}

    f = bsd.min_start_fill_month(1, float(IMPORT_CEILING[1]), wc_func, peak_demand, residual_demand)
    # Run full ceiling obligations table month by month
    ceiling_obligations = {}
    for m in bsd.MONTH_ORDER:
        fi = bsd.min_start_fill_month(m, float(IMPORT_CEILING[m]),
                                       wc_func, peak_demand, residual_demand)
        ceiling_obligations[bsd.MONTH_NAMES[m]] = (
            round(fi * bsd.CAPACITY_TWH / 100, 2) if fi is not None else None
        )
    break  # ceiling is import-assumption-driven, not scenario-driven

ceiling_df = pd.DataFrame([ceiling_obligations], index=["Obligation (TWh)"]).T
print(f"Ceiling assumption: {IMPORT_ASSUMPTION}")
print(f"P99 cold-day imports used:")
print(IMPORT_CEILING.rename(bsd.MONTH_NAMES).round(1).to_string())
print()
ceiling_df

---
## 8. Caveats

1. **Monthly independence.** Each obligation is a standalone 30-day stress test.
2. **Deterministic imports.** Month-specific percentiles applied as constants — conservative upper-bound.
3. **Withdrawal curve at low fill (<20%).** Poorly identified empirically; blend mitigates this.
4. **Demand/import correlation.** Commercially driven (not infrastructural); model is conservative on imports.
5. **End-of-season reserve.** ~0.5 TWh at Mar 31 recommended as a separate standalone instrument.
6. **Czech monthly framing vs. EU floating window.** Conservative and administratively tractable.